In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
df = pd.read_excel("im_bank_data.xlsx")

In [4]:
# 1-1단계
# 원본 보존 + 패널 키 중복 여부만 확인

clean_panel = df.copy(deep=True)

duplicate_count = clean_panel.duplicated(
    subset=["기준년월", "법인ID"]
).sum()

print("전체 행 수:", len(clean_panel))
print("법인 수:", clean_panel["법인ID"].nunique())
print("기준년월 수:", clean_panel["기준년월"].nunique())
print("법인ID × 기준년월 중복 행 수:", duplicate_count)

전체 행 수: 365988
법인 수: 15473
기준년월 수: 36
법인ID × 기준년월 중복 행 수: 0


In [5]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [6]:
# clean_panel은 원본 복사본으로 유지
clean_panel = df.copy(deep=True)

# 자료형별 컬럼 확인
dtype_summary = pd.DataFrame({
    "dtype": clean_panel.dtypes.astype(str),
    "결측수": clean_panel.isna().sum(),
    "고유값수": clean_panel.nunique(dropna=True)
})

display(dtype_summary)

,dtype,결측수,고유값수
기준년월,int64,0,36
법인ID,str,0,15473
업종_대분류,str,0,18
업종_중분류,str,0,61
사업장_시도,str,12424,5
사업장_시군구,str,12424,13
법인_고객등급,str,0,3
전담고객여부,str,0,2
요구불예금잔액,float64,0,562
거치식예금잔액,float64,0,479


In [7]:
object_cols = clean_panel.select_dtypes(include=["object"]).columns.tolist()

print("문자열(object) 컬럼 수:", len(object_cols))

for col in object_cols:
    print(col)

문자열(object) 컬럼 수: 36
법인ID
업종_대분류
업종_중분류
사업장_시도
사업장_시군구
법인_고객등급
전담고객여부
요구불예금좌수
거치식예금좌수
적립식예금좌수
수익증권좌수
신탁좌수
퇴직연금좌수
여신_운전자금대출좌수
운전_할인어음좌수
운전_당좌대출좌수
운전_일반자금대출좌수
운전_무역금융좌수
운전_주택자금대출좌수
운전_기업구매자금대출좌수
운전_외상매출채권담보대출좌수
운전_기타운전자금대출좌수
여신_시설자금대출좌수
시설_일반자금대출좌수
시설_에너지절약시설대출좌수
시설_주택자금대출좌수
시설_기타시설자금대출좌수
신용카드개수
외환_수출실적거래건수
외환_수입실적거래건수
창구거래건수
인터넷뱅킹거래건수
스마트뱅킹거래건수
폰뱅킹거래건수
ATM거래건수
자동이체거래건수


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_19092\1205333809.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = clean_panel.select_dtypes(include=["object"]).columns.tolist()


In [8]:
# 1-3단계
# 문자열 컬럼을
# ① 순서형 좌수/건수
# ② 일반 범주형
# 으로 구분만 해보기

object_cols = clean_panel.select_dtypes(include=["object"]).columns.tolist()

ordinal_cols = [
    c for c in object_cols
    if (
        "좌수" in c
        or "거래건수" in c
        or c == "신용카드개수"
    )
]

categorical_cols = [
    c for c in object_cols
    if c not in ordinal_cols
]

print("전체 문자열 컬럼:", len(object_cols))
print("순서형 좌수/건수 컬럼:", len(ordinal_cols))
print("일반 범주형 컬럼:", len(categorical_cols))

print("\n[일반 범주형]")
for c in categorical_cols:
    print("-", c)

print("\n[순서형 좌수/건수]")
for c in ordinal_cols:
    print("-", c)

전체 문자열 컬럼: 36
순서형 좌수/건수 컬럼: 29
일반 범주형 컬럼: 7

[일반 범주형]
- 법인ID
- 업종_대분류
- 업종_중분류
- 사업장_시도
- 사업장_시군구
- 법인_고객등급
- 전담고객여부

[순서형 좌수/건수]
- 요구불예금좌수
- 거치식예금좌수
- 적립식예금좌수
- 수익증권좌수
- 신탁좌수
- 퇴직연금좌수
- 여신_운전자금대출좌수
- 운전_할인어음좌수
- 운전_당좌대출좌수
- 운전_일반자금대출좌수
- 운전_무역금융좌수
- 운전_주택자금대출좌수
- 운전_기업구매자금대출좌수
- 운전_외상매출채권담보대출좌수
- 운전_기타운전자금대출좌수
- 여신_시설자금대출좌수
- 시설_일반자금대출좌수
- 시설_에너지절약시설대출좌수
- 시설_주택자금대출좌수
- 시설_기타시설자금대출좌수
- 신용카드개수
- 외환_수출실적거래건수
- 외환_수입실적거래건수
- 창구거래건수
- 인터넷뱅킹거래건수
- 스마트뱅킹거래건수
- 폰뱅킹거래건수
- ATM거래건수
- 자동이체거래건수


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_19092\121234259.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = clean_panel.select_dtypes(include=["object"]).columns.tolist()


In [9]:
# 2-1단계: 법인별 관측공백 구조 확인
# 아직 삭제/보간/0대체 안 함

import pandas as pd

gap_check = clean_panel[["법인ID", "기준년월"]].copy()

# 월 계산을 위해서만 임시 Period 변수 생성
gap_check["월"] = pd.to_datetime(
    gap_check["기준년월"].astype(str),
    format="%Y%m"
).dt.to_period("M")

gap_check = gap_check.sort_values(["법인ID", "월"])

# 다음 관측월
gap_check["다음관측월"] = (
    gap_check.groupby("법인ID")["월"].shift(-1)
)

# 현재 월과 다음 관측월 사이에 빠진 개월 수
gap_check["공백개월수"] = (
    gap_check["다음관측월"].apply(
        lambda x: x.ordinal if pd.notna(x) else pd.NA
    )
    - gap_check["월"].apply(lambda x: x.ordinal)
    - 1
)

# 중간 공백이 있는 행만
gap_events = gap_check[
    gap_check["공백개월수"].fillna(0) > 0
].copy()

print("공백 발생 이벤트 수:", len(gap_events))
print("공백 경험 법인 수:", gap_events["법인ID"].nunique())

print("\n공백 길이별 이벤트 수:")
display(
    gap_events["공백개월수"]
    .value_counts()
    .sort_index()
    .rename_axis("공백개월수")
    .to_frame("이벤트수")
)

공백 발생 이벤트 수: 13084
공백 경험 법인 수: 6416

공백 길이별 이벤트 수:


,이벤트수
공백개월수,
1,11039
2,1231
3,257
4,122
5,68
6,43
7,42
8,38
9,25


In [10]:
# 2-2단계: 법인별 공백 이벤트 횟수 확인

gap_count_by_corp = (
    gap_events
    .groupby("법인ID")
    .size()
    .rename("공백이벤트수")
    .reset_index()
)

print("공백 경험 법인 수:", len(gap_count_by_corp))

print("\n법인별 공백 이벤트 횟수 분포:")
display(
    gap_count_by_corp["공백이벤트수"]
    .value_counts()
    .sort_index()
    .rename_axis("공백이벤트수")
    .to_frame("법인수")
)

공백 경험 법인 수: 6416

법인별 공백 이벤트 횟수 분포:


,법인수
공백이벤트수,
1,3373
2,1472
3,716
4,363
5,203
6,121
7,52
8,41
9,36


In [11]:
# 2-3단계: 공백이 잦은 법인의 관측 구조 확인

# 법인별 공백 요약
gap_summary = (
    gap_events.groupby("법인ID")
    .agg(
        공백이벤트수=("공백개월수", "size"),
        최대공백개월수=("공백개월수", "max"),
        평균공백개월수=("공백개월수", "mean")
    )
    .reset_index()
)

# 법인별 실제 관측 개월 수
observed_months = (
    clean_panel.groupby("법인ID")
    .size()
    .rename("관측개월수")
    .reset_index()
)

gap_summary = gap_summary.merge(
    observed_months,
    on="법인ID",
    how="left"
)

# 공백 이벤트 많은 순으로 확인
gap_summary = gap_summary.sort_values(
    ["공백이벤트수", "최대공백개월수"],
    ascending=[False, False]
)

display(gap_summary.head(100))

In [12]:
# 2-4단계: 공백 길이 분포 확인

gap_length_summary = (
    gap_events["공백개월수"]
    .value_counts()
    .sort_index()
    .rename_axis("공백개월수")
    .to_frame("이벤트수")
)

gap_length_summary["비율"] = (
    gap_length_summary["이벤트수"]
    / gap_length_summary["이벤트수"].sum()
)

display(gap_length_summary)

# 보기 쉽게 1개월 / 2개월 / 3개월 이상으로 묶기
gap_events["공백구간"] = pd.cut(
    gap_events["공백개월수"],
    bins=[0, 1, 2, float("inf")],
    labels=["1개월", "2개월", "3개월 이상"]
)

display(
    gap_events["공백구간"]
    .value_counts()
    .reindex(["1개월", "2개월", "3개월 이상"])
    .to_frame("이벤트수")
)

,이벤트수,비율
공백개월수,,
1,11039,0.843702
2,1231,0.094084
3,257,0.019642
4,122,0.009324
5,68,0.005197
6,43,0.003286
7,42,0.003210
8,38,0.002904
9,25,0.001911


,이벤트수
공백구간,
1개월,11039
2개월,1231
3개월 이상,814


In [13]:
# 공백 길이별 '고유 법인 수'

gap_firm_summary = (
    gap_events
    .groupby("공백개월수")["법인ID"]
    .nunique()
    .to_frame("법인수")
)

display(gap_firm_summary)

,법인수
공백개월수,
1,5858
2,981
3,232
4,118
5,67
6,43
7,41
8,37
9,25


In [14]:
# 기준월별 분석 가능 표본 수 확인
# T = 2024-01 ~ 2025-06
#
# 1) T에 있는 법인
# 2) 그중 T-12에도 있는 법인
# 3) 그중 T+1 ~ T+6에서 몇 개월 관측되는지 계산

import pandas as pd

# 계산용 데이터만 따로 생성
obs = clean_panel[["법인ID", "기준년월"]].copy()

# 월 계산을 편하게 하기 위한 임시 변수
obs["월"] = pd.to_datetime(
    obs["기준년월"].astype(str),
    format="%Y%m"
).dt.to_period("M")


# --------------------------------------------------
# 법인별 관측 월을 빠르게 확인하기 위한 집합
# --------------------------------------------------

month_to_ids = {
    month: set(group["법인ID"])
    for month, group in obs.groupby("월")
}


# --------------------------------------------------
# 기준월
# --------------------------------------------------

reference_months = pd.period_range(
    "2024-01",
    "2025-06",
    freq="M"
)


summary_rows = []
followup_distribution_rows = []


for T in reference_months:

    # -------------------------------
    # 1. 기준월 T에 존재하는 법인
    # -------------------------------

    ids_T = month_to_ids.get(T, set())


    # -------------------------------
    # 2. 전년동월 T-12에도 존재하는 법인
    # -------------------------------

    prev_month = T - 12
    ids_prev = month_to_ids.get(prev_month, set())

    # T와 T-12 모두 관측된 법인
    eligible_ids = ids_T & ids_prev


    # -------------------------------
    # 3. 후속 6개월 관측 개월 수
    # -------------------------------

    followup_counts = {}

    for corp_id in eligible_ids:

        observed_count = 0

        for k in range(1, 7):
            future_month = T + k

            if corp_id in month_to_ids.get(future_month, set()):
                observed_count += 1

        followup_counts[corp_id] = observed_count


    # -------------------------------
    # 4. 기준월별 요약
    # -------------------------------

    n_T = len(ids_T)
    n_eligible = len(eligible_ids)

    n_6 = sum(v == 6 for v in followup_counts.values())
    n_5plus = sum(v >= 5 for v in followup_counts.values())
    n_4plus = sum(v >= 4 for v in followup_counts.values())

    summary_rows.append({
        "기준월": str(T),

        "T_관측법인수": n_T,

        "T_minus_12_동시관측수": n_eligible,

        "전년동월_미관측으로_제외": n_T - n_eligible,

        "전년동월_관측가능률":
            n_eligible / n_T if n_T > 0 else None,

        "후속6개월_6개관측": n_6,

        "후속6개월_5개이상관측": n_5plus,

        "후속6개월_4개이상관측": n_4plus,

        "6개완전관측_유지율":
            n_6 / n_eligible if n_eligible > 0 else None,

        "5개이상관측_유지율":
            n_5plus / n_eligible if n_eligible > 0 else None,

        "4개이상관측_유지율":
            n_4plus / n_eligible if n_eligible > 0 else None
    })


    # -------------------------------
    # 5. 후속 관측개월수 0~6 분포
    # -------------------------------

    for n_months in range(7):

        count = sum(
            v == n_months
            for v in followup_counts.values()
        )

        followup_distribution_rows.append({
            "기준월": str(T),
            "후속6개월중_관측개월수": n_months,
            "법인수": count
        })


# --------------------------------------------------
# 결과표
# --------------------------------------------------

availability_summary = pd.DataFrame(summary_rows)

followup_distribution = pd.DataFrame(
    followup_distribution_rows
)


display(availability_summary)

,기준월,T_관측법인수,T_minus_12_동시관측수,전년동월_미관측으로_제외,전년동월_관측가능률,후속6개월_6개관측,후속6개월_5개이상관측,후속6개월_4개이상관측,6개완전관측_유지율,5개이상관측_유지율,4개이상관측_유지율
0,2024-01,10145,8513,1632,0.839133,6320,7542,7935,0.742394,0.885939,0.932104
1,2024-02,10057,8315,1742,0.826787,6247,7333,7731,0.751293,0.881900,0.929765
2,2024-03,9667,7934,1733,0.820730,6267,7167,7406,0.789892,0.903327,0.933451
3,2024-04,10193,8402,1791,0.824291,6552,7494,7792,0.779814,0.891930,0.927398
4,2024-05,10250,8418,1832,0.821268,6641,7530,7847,0.788905,0.894512,0.932169
5,2024-06,9914,8151,1763,0.822171,6636,7458,7668,0.814133,0.914980,0.940743
6,2024-07,10318,8502,1816,0.823997,6812,7660,7970,0.801223,0.900964,0.937426
7,2024-08,10374,8516,1858,0.820898,6818,7699,7993,0.800611,0.904063,0.938586
8,2024-09,10161,8154,2007,0.802480,6585,7485,7746,0.807579,0.917954,0.949963
9,2024-10,10411,8496,1915,0.816060,6670,7686,8022,0.785075,0.904661,0.944209


In [15]:
followup_pivot = (
    followup_distribution
    .pivot(
        index="기준월",
        columns="후속6개월중_관측개월수",
        values="법인수"
    )
    .fillna(0)
    .astype(int)
)

followup_pivot.columns = [
    f"{c}개월관측"
    for c in followup_pivot.columns
]

display(followup_pivot)

,0개월관측,1개월관측,2개월관측,3개월관측,4개월관측,5개월관측,6개월관측
기준월,,,,,,,
2024-01,95,136,135,212,393,1222,6320
2024-02,138,117,159,170,398,1086,6247
2024-03,105,129,124,170,239,900,6267
2024-04,150,152,164,144,298,942,6552
2024-05,148,156,123,144,317,889,6641
2024-06,137,90,117,139,210,822,6636
2024-07,115,131,136,150,310,848,6812
2024-08,126,128,124,145,294,881,6818
2024-09,100,88,106,114,261,900,6585


In [16]:
availability_summary.to_clipboard(index=False, excel=True)

In [17]:
followup_pivot.to_clipboard(index=True, excel=True)

In [18]:
# 3-1단계: 위축 신호 정의 검증
# Case 1:
# - T, T-12 실제 관측 필수
# - before(T-14~T-12), after(T-2~T) 각각 최소 2/3 관측
# - 가용 관측값 중앙값 사용
#
# Case 2:
# - before, after 모두 3/3 완전관측
#
# 공통:
# - 요구불입금금액 기준
# - 전년 대비 30% 이상 감소 = 위축 신호
# - 결측값 대체/외삽 없음

import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. 계산용 데이터
# --------------------------------------------------

signal_base = clean_panel[
    ["법인ID", "기준년월", "요구불입금금액"]
].copy()

signal_base["월"] = pd.to_datetime(
    signal_base["기준년월"].astype(str),
    format="%Y%m"
).dt.to_period("M")


# --------------------------------------------------
# 2. 법인 × 월 형태로 변환
# --------------------------------------------------

# 실제 금액
amount_panel = signal_base.pivot(
    index="법인ID",
    columns="월",
    values="요구불입금금액"
)

# 행 자체가 존재했는지 확인용
presence_base = signal_base[
    ["법인ID", "월"]
].copy()

presence_base["관측"] = True

presence_panel = (
    presence_base
    .pivot(
        index="법인ID",
        columns="월",
        values="관측"
    )
    .fillna(False)
)


# --------------------------------------------------
# 3. 기준월
# T-14가 존재하고, 이후 T+6까지 추적 가능한 범위
# --------------------------------------------------

reference_months = pd.period_range(
    "2024-03",
    "2025-06",
    freq="M"
)


signal_event_list = []


# --------------------------------------------------
# 4. 기준월별 Case 1 / Case 2 계산
# --------------------------------------------------

for T in reference_months:

    before_months = [T - 14, T - 13, T - 12]
    after_months  = [T - 2, T - 1, T]

    # 각 창의 실제 관측 개월 수
    before_n = presence_panel[before_months].sum(axis=1)
    after_n  = presence_panel[after_months].sum(axis=1)

    # T와 T-12 자체는 반드시 실제 관측
    anchor_ok = (
        presence_panel[T]
        & presence_panel[T - 12]
    )

    # 가용 관측값 중앙값
    before_median = amount_panel[before_months].median(
        axis=1,
        skipna=True
    )

    after_median = amount_panel[after_months].median(
        axis=1,
        skipna=True
    )

    # -------------------------------
    # Case 1: 각 창 2/3 이상
    # -------------------------------

    case1_obs_ok = (
        anchor_ok
        & (before_n >= 2)
        & (after_n >= 2)
    )

    # 감소율 분모가 0이면 계산 불가
    case1_calc_ok = (
        case1_obs_ok
        & (before_median > 0)
    )

    case1_rate = (
        (before_median - after_median)
        / before_median
    )

    case1_signal = (
        case1_calc_ok
        & (case1_rate >= 0.30)
    )


    # -------------------------------
    # Case 2: 3/3 완전관측
    # -------------------------------

    case2_obs_ok = (
        anchor_ok
        & (before_n == 3)
        & (after_n == 3)
    )

    case2_calc_ok = (
        case2_obs_ok
        & (before_median > 0)
    )

    case2_signal = (
        case2_calc_ok
        & (case1_rate >= 0.30)
    )


    # 내부 분석용 이벤트 테이블
    tmp = pd.DataFrame({
        "법인ID": amount_panel.index,
        "기준월": str(T),

        "before_관측개월수": before_n.values,
        "after_관측개월수": after_n.values,

        "before_중앙값": before_median.values,
        "after_중앙값": after_median.values,
        "위축률": case1_rate.values,

        "Case1_관측조건": case1_obs_ok.values,
        "Case1_계산가능": case1_calc_ok.values,
        "Case1_30퍼신호": case1_signal.values,

        "Case2_관측조건": case2_obs_ok.values,
        "Case2_계산가능": case2_calc_ok.values,
        "Case2_30퍼신호": case2_signal.values
    })

    signal_event_list.append(tmp)


signal_events = pd.concat(
    signal_event_list,
    ignore_index=True
)


# --------------------------------------------------
# 5. 기준월별 집계 결과만 출력
# --------------------------------------------------

signal_summary = (
    signal_events
    .groupby("기준월")
    .agg(
        Case1_관측조건충족=("Case1_관측조건", "sum"),
        Case1_계산가능=("Case1_계산가능", "sum"),
        Case1_30퍼신호=("Case1_30퍼신호", "sum"),

        Case2_완전관측=("Case2_관측조건", "sum"),
        Case2_계산가능=("Case2_계산가능", "sum"),
        Case2_30퍼신호=("Case2_30퍼신호", "sum")
    )
    .reset_index()
)

signal_summary["Case2_표본유지율"] = (
    signal_summary["Case2_계산가능"]
    / signal_summary["Case1_계산가능"]
)

display(signal_summary)

,기준월,Case1_관측조건충족,Case1_계산가능,Case1_30퍼신호,Case2_완전관측,Case2_계산가능,Case2_30퍼신호,Case2_표본유지율
0,2024-03,7788,7289,1976,7015,6570,1729,0.901358
1,2024-04,8196,7619,2079,6963,6511,1673,0.854574
2,2024-05,8171,7634,2146,7002,6589,1774,0.863112
3,2024-06,7979,7460,2053,7176,6710,1786,0.899464
4,2024-07,8257,7688,2234,7196,6735,1824,0.876041
5,2024-08,8342,7766,2280,7293,6834,1904,0.879990
6,2024-09,8033,7509,2041,7346,6863,1815,0.913970
7,2024-10,8291,7717,2254,7366,6886,1900,0.892316
8,2024-11,8349,7728,2264,7347,6835,1904,0.884446
9,2024-12,8065,7526,2140,7386,6875,1887,0.913500


In [19]:
# 3-2단계: 위축 임계값 민감도 분석
# 20% / 30% / 40% 비교

thresholds = {
    "20퍼": 0.20,
    "30퍼": 0.30,
    "40퍼": 0.40
}

# --------------------------------------------------
# 1. Case 1 / Case 2별 신호 생성
# --------------------------------------------------

for name, threshold in thresholds.items():

    signal_events[f"Case1_{name}신호"] = (
        signal_events["Case1_계산가능"]
        & (signal_events["위축률"] >= threshold)
    )

    signal_events[f"Case2_{name}신호"] = (
        signal_events["Case2_계산가능"]
        & (signal_events["위축률"] >= threshold)
    )


# --------------------------------------------------
# 2. 기준월별 신호 수
# --------------------------------------------------

threshold_summary = (
    signal_events
    .groupby("기준월")
    .agg(
        Case1_계산가능=("Case1_계산가능", "sum"),
        Case1_20퍼=("Case1_20퍼신호", "sum"),
        Case1_30퍼=("Case1_30퍼신호", "sum"),
        Case1_40퍼=("Case1_40퍼신호", "sum"),

        Case2_계산가능=("Case2_계산가능", "sum"),
        Case2_20퍼=("Case2_20퍼신호", "sum"),
        Case2_30퍼=("Case2_30퍼신호", "sum"),
        Case2_40퍼=("Case2_40퍼신호", "sum")
    )
    .reset_index()
)

# 각 기준에서 계산가능 표본 중 몇 %가 위축신호인지
for case in ["Case1", "Case2"]:
    for pct in ["20퍼", "30퍼", "40퍼"]:
        threshold_summary[f"{case}_{pct}_신호율"] = (
            threshold_summary[f"{case}_{pct}"]
            / threshold_summary[f"{case}_계산가능"]
        )

display(threshold_summary)

,기준월,Case1_계산가능,Case1_20퍼,Case1_30퍼,Case1_40퍼,Case2_계산가능,Case2_20퍼,Case2_30퍼,Case2_40퍼,Case1_20퍼_신호율,Case1_30퍼_신호율,Case1_40퍼_신호율,Case2_20퍼_신호율,Case2_30퍼_신호율,Case2_40퍼_신호율
0,2024-03,7289,2503,1976,1579,6570,2214,1729,1362,0.343394,0.271093,0.216628,0.336986,0.263166,0.207306
1,2024-04,7619,2542,2079,1701,6511,2089,1673,1342,0.333640,0.272870,0.223258,0.320842,0.256950,0.206113
2,2024-05,7634,2638,2146,1741,6589,2210,1774,1423,0.345559,0.281111,0.228059,0.335407,0.269237,0.215966
3,2024-06,7460,2555,2053,1655,6710,2230,1786,1433,0.342493,0.275201,0.221850,0.332340,0.266170,0.213562
4,2024-07,7688,2730,2234,1842,6735,2265,1824,1479,0.355099,0.290583,0.239594,0.336303,0.270824,0.219599
5,2024-08,7766,2820,2280,1874,6834,2394,1904,1534,0.363121,0.293587,0.241308,0.350307,0.278607,0.224466
6,2024-09,7509,2518,2041,1640,6863,2249,1815,1450,0.335331,0.271807,0.218405,0.327699,0.264462,0.211278
7,2024-10,7717,2795,2254,1824,6886,2392,1900,1522,0.362187,0.292082,0.236361,0.347371,0.275922,0.221028
8,2024-11,7728,2821,2264,1842,6835,2414,1904,1533,0.365036,0.292961,0.238354,0.353182,0.278566,0.224287
9,2024-12,7526,2638,2140,1738,6875,2355,1887,1514,0.350518,0.284348,0.230933,0.342545,0.274473,0.220218


In [20]:
# Case1 30% 위축 신호가 잡힌 법인×시점에서, T+1~T+6 중 전년동월과 짝이 맞는 달이 몇 개인가?

# 3-3단계: Case1 30% 위축군의 후속 6개월 matched pair 분포 확인

import pandas as pd

# Case1 30% 위축 신호만
signal_30 = signal_events.loc[
    signal_events["Case1_30퍼신호"],
    ["법인ID", "기준월"]
].copy()

signal_30["T"] = pd.PeriodIndex(
    signal_30["기준월"],
    freq="M"
)

result = []

for row in signal_30.itertuples(index=False):

    corp = row.법인ID
    T = row.T

    pair_count = 0
    followup_obs_count = 0

    # T+1 ~ T+6 각각 전년동월과 pair 확인
    for k in range(1, 7):

        current_month = T + k
        prev_year_month = T + k - 12

        current_obs = bool(
            presence_panel.loc[corp, current_month]
        )

        prev_year_obs = bool(
            presence_panel.loc[corp, prev_year_month]
        )

        # 후속월 자체 관측 개수
        followup_obs_count += int(current_obs)

        # 현재월 + 전년동월 둘 다 있을 때만 matched pair
        pair_count += int(
            current_obs and prev_year_obs
        )

    # T+4, T+5, T+6이 모두 미관측인지
    last3_all_missing = all(
        not bool(presence_panel.loc[corp, T + k])
        for k in [4, 5, 6]
    )

    result.append({
        "법인ID": corp,
        "기준월": str(T),
        "후속6개월_관측수": followup_obs_count,
        "matched_pair수": pair_count,
        "마지막3개월_연속미관측": last3_all_missing
    })


signal_followup_check = pd.DataFrame(result)


# matched pair 0~6개 분포
followup_pair_summary = (
    signal_followup_check
    .groupby("matched_pair수")
    .size()
    .reindex(range(7), fill_value=0)
    .rename("법인×신호수")
    .reset_index()
)

followup_pair_summary["비율"] = (
    followup_pair_summary["법인×신호수"]
    / followup_pair_summary["법인×신호수"].sum()
)

display(followup_pair_summary)


# 추가 확인: 마지막 3개월 연속 미관측 비율
display(
    signal_followup_check[
        "마지막3개월_연속미관측"
    ].value_counts(dropna=False)
)

,matched_pair수,법인×신호수,비율
0,0,918,0.026124
1,1,914,0.026010
2,2,973,0.027689
3,3,1161,0.033039
4,4,2416,0.068754
5,5,5977,0.170091
6,6,22781,0.648293


마지막3개월_연속미관측
False    32119
True      3021
Name: count, dtype: int64

In [24]:
# 4-1단계: 요구불입금금액의 월별 계절성 확인
# 법인별 거래규모 차이를 제거한 뒤, 월별 상대적 수준을 확인

season_check = clean_panel[
    ["법인ID", "기준년월", "업종_대분류", "요구불입금금액"]
].copy()

# 연도 / 월 생성
season_check["연도"] = (
    season_check["기준년월"] // 100
)

season_check["월"] = (
    season_check["기준년월"] % 100
)

# 금액 분포가 큰 편이므로 log1p 변환
season_check["log_입금"] = np.log1p(
    season_check["요구불입금금액"]
)

# 법인별 평소 거래수준 제거
season_check["법인평균_log입금"] = (
    season_check
    .groupby("법인ID")["log_입금"]
    .transform("mean")
)

season_check["평소대비_차이"] = (
    season_check["log_입금"]
    - season_check["법인평균_log입금"]
)

# 달별 대표적인 상대수준
monthly_season = (
    season_check
    .groupby("월")
    .agg(
        중앙값=("평소대비_차이", "median"),
        평균=("평소대비_차이", "mean"),
        관측수=("법인ID", "size")
    )
    .reset_index()
)

display(monthly_season)

monthly_season["중앙값_퍼센트해석"] = (
    np.exp(monthly_season["중앙값"]) - 1
) * 100

monthly_season["평균_퍼센트해석"] = (
    np.exp(monthly_season["평균"]) - 1
) * 100

display(monthly_season)

,월,중앙값,평균,관측수
0,1,0.000000,-0.007427,30967
1,2,-0.037019,-0.055803,30501
2,3,-0.013104,0.012603,29473
3,4,0.000000,0.015272,30541
4,5,0.000000,0.007406,30648
5,6,-0.006766,0.021301,29885
6,7,0.000000,-0.001084,30857
7,8,-0.043579,-0.066175,30959
8,9,0.000000,0.018670,30071
9,10,-0.014850,-0.051897,31001


,월,중앙값,평균,관측수,중앙값_퍼센트해석,평균_퍼센트해석
0,1,0.000000,-0.007427,30967,0.000000,-0.739981
1,2,-0.037019,-0.055803,30501,-3.634193,-5.427445
2,3,-0.013104,0.012603,29473,-1.301845,1.268230
3,4,0.000000,0.015272,30541,0.000000,1.538941
4,5,0.000000,0.007406,30648,0.000000,0.743303
5,6,-0.006766,0.021301,29885,-0.674279,2.152992
6,7,0.000000,-0.001084,30857,0.000000,-0.108318
7,8,-0.043579,-0.066175,30959,-4.264330,-6.403292
8,9,0.000000,0.018670,30071,0.000000,1.884555
9,10,-0.014850,-0.051897,31001,-1.473984,-5.057366


In [25]:
# 연도별로 같은 월 패턴이 반복되는지 확인

year_month_season = (
    season_check
    .groupby(["연도", "월"])
    .agg(
        중앙값=("평소대비_차이", "median")
    )
    .reset_index()
)

year_month_season["중앙값_퍼센트해석"] = (
    np.exp(year_month_season["중앙값"]) - 1
) * 100

season_pivot = (
    year_month_season
    .pivot(
        index="월",
        columns="연도",
        values="중앙값_퍼센트해석"
    )
)

display(season_pivot)

연도,2023,2024,2025
월,,,
1,0.417667,2.249377,-7.428545
2,-1.511438,-1.323869,-8.369139
3,4.354291,-4.077886,-5.192249
4,-0.476017,3.363931,-1.928892
5,5.892854,1.146144,-6.534112
6,3.605163,-4.568534,-2.807971
7,0.000000,3.108965,-3.289028
8,0.525992,-4.613928,-10.325913
9,0.000000,0.000000,0.000000


In [26]:
# 연도별 월 패턴 상관계수
season_corr = season_pivot.corr()

display(season_corr)

연도,2023,2024,2025
연도,,,
2023,1.000000,0.305356,0.493745
2024,0.305356,1.000000,0.729447
2025,0.493745,0.729447,1.000000


In [27]:
# 4-2단계: 연도 전체 수준을 제거한 뒤 월별 패턴 확인

# 각 연도의 전체적인 수준 제거
season_check["연도기준_차이"] = (
    season_check["평소대비_차이"]
    - season_check
        .groupby("연도")["평소대비_차이"]
        .transform("median")
)

# 연도 × 월 중앙값
year_month_adjusted = (
    season_check
    .groupby(["연도", "월"])
    .agg(
        중앙값=("연도기준_차이", "median")
    )
    .reset_index()
)

# 퍼센트 해석
year_month_adjusted["퍼센트해석"] = (
    np.exp(year_month_adjusted["중앙값"]) - 1
) * 100

# 보기 좋게 피벗
adjusted_pivot = (
    year_month_adjusted
    .pivot(
        index="월",
        columns="연도",
        values="퍼센트해석"
    )
)

display(adjusted_pivot)

연도,2023,2024,2025
월,,,
1,-1.011947,2.249377,-3.152908
2,-2.913588,-1.323869,-4.136945
3,2.868633,-4.077886,-0.813323
4,-1.892908,3.363931,2.600760
5,4.385292,1.146144,-2.217164
6,2.130170,-4.568534,1.681078
7,-1.423667,3.108965,1.177802
8,-0.905164,-4.613928,-6.184098
9,-1.423667,0.000000,4.618742


In [23]:
# 4-2단계: 업종_대분류별 월 계절성 확인

industry_season = clean_panel[
    ["법인ID", "기준년월", "업종_대분류", "요구불입금금액"]
].copy()

# 월 생성
industry_season["월"] = (
    industry_season["기준년월"] % 100
)

# 금액 규모 차이를 줄이기 위해 log 변환
industry_season["log_입금"] = np.log1p(
    industry_season["요구불입금금액"]
)

# 법인별 자기 평균 수준
industry_season["법인평균_log입금"] = (
    industry_season
    .groupby("법인ID")["log_입금"]
    .transform("mean")
)

# 자기 평소 수준 대비 해당 월의 상대적 차이
industry_season["평소대비_차이"] = (
    industry_season["log_입금"]
    - industry_season["법인평균_log입금"]
)

# 업종 × 월별 집계
industry_month_season = (
    industry_season
    .groupby(["업종_대분류", "월"])
    .agg(
        중앙값=("평소대비_차이", "median"),
        평균=("평소대비_차이", "mean"),
        관측수=("법인ID", "size")
    )
    .reset_index()
)

# 중앙값을 사람이 보기 쉬운 %로 변환
industry_month_season["중앙값_퍼센트"] = (
    np.expm1(industry_month_season["중앙값"]) * 100
)

# ------------------------------------------
# 1. 업종별 1~12월 계절 패턴
# ------------------------------------------

industry_season_pivot = (
    industry_month_season
    .pivot(
        index="업종_대분류",
        columns="월",
        values="중앙값_퍼센트"
    )
    .round(1)
)

display(industry_season_pivot)


# ------------------------------------------
# 2. 업종별 월간 최대 차이
# ------------------------------------------

industry_season_range = (
    industry_season_pivot.max(axis=1)
    - industry_season_pivot.min(axis=1)
).sort_values(ascending=False)

display(
    industry_season_range
    .rename("월별_최대차이_%p")
    .to_frame()
)

월,1,2,3,4,5,6,7,8,9,10,11,12
업종_대분류,,,,,,,,,,,,
건설업,0.0,-13.3,2.5,0.0,-0.4,9.0,0.0,-7.8,12.2,-9.5,-5.2,72.2
광업,1.3,-2.9,-24.5,1.3,6.9,0.5,3.1,-11.6,9.2,-4.9,-16.7,5.9
교육 서비스업,-5.9,-3.0,-1.9,-1.6,-0.2,-2.6,1.4,-4.6,-5.4,-8.9,-4.5,3.7
금융 및 보험업,-4.2,0.0,-1.4,0.0,-6.2,0.0,-3.1,-12.1,-0.6,-3.5,-6.7,0.0
"농업, 임업 및 어업",-5.1,-5.5,-0.8,-4.6,0.2,-3.7,-5.8,-1.6,-3.9,-5.1,-7.9,-9.9
도매 및 소매업,0.6,-3.0,-0.2,0.8,1.0,-1.8,0.1,-4.0,-0.0,-0.4,-3.2,3.1
보건업 및 사회복지 서비스업,0.0,-0.1,0.3,-0.2,-4.0,-1.4,-0.3,1.8,0.0,1.6,-2.2,3.4
부동산업,-2.0,-2.1,-4.3,-4.7,-6.5,-8.5,-2.2,-6.4,-3.2,-5.1,-9.2,-0.0
"사업시설 관리, 사업 지원 및 임대 서비스업",-3.2,-5.0,-2.4,-0.9,0.0,-2.3,-1.8,-4.2,-3.3,0.9,0.8,4.4


,월별_최대차이_%p
업종_대분류,
건설업,85.5
광업,33.7
"전기, 가스, 증기 및 공기조절 공급업",31.3
"전문, 과학 및 기술 서비스업",26.4
"협회 및 단체, 수리 및 기타 개인 서비스업",20.8
정보통신업,20.2
"수도, 하수 및 폐기물 처리, 원료 재생업",19.5
제조업,13.8
교육 서비스업,12.6
